# Clase 130 — Arquitecturas CNN: de LeNet-5 a ConvNeXt

Recorrido por la evolución de las CNN — **LeNet-5 (1998)** → AlexNet → VGG →
GoogLeNet (Inception) → **ResNet (skip connections)** → Xception
(depthwise-separable) → SENet (squeeze-and-excite) → EfficientNet (compound
scaling) → **ConvNeXt (2022)**. Tres patrones clave: profundidad creciente,
módulos con paths múltiples y eficiencia paramétrica. Aquí construimos los
bloques canónicos en Keras.

Requiere: `tensorflow` / `keras` (≥ 3.0). No se ejecuta aquí; código idiomático
y correcto por API.

## 1. LeNet-5 (1998): la cuna de las CNN

In [ ]:
from tensorflow import keras
from tensorflow.keras import layers
keras.utils.set_random_seed(42)

lenet = keras.Sequential([
    keras.Input(shape=(32, 32, 1)),
    layers.Conv2D(6, 5, activation="tanh", padding="valid"),    # C1
    layers.AveragePooling2D(2),                                 # S2
    layers.Conv2D(16, 5, activation="tanh", padding="valid"),   # C3
    layers.AveragePooling2D(2),                                 # S4
    layers.Flatten(),
    layers.Dense(120, activation="tanh"),                       # C5
    layers.Dense(84, activation="tanh"),                        # F6
    layers.Dense(10, activation="softmax"),                     # salida
], name="LeNet5")
print("LeNet-5 params:", lenet.count_params())

## 2. Bloque tipo VGG: solo conv 3×3 + maxpool

VGG es muy uniforme: apila convs `3×3` con `padding='same'` seguidas de un
`MaxPooling2D(2)`.

In [ ]:
def bloque_vgg(x, filtros, n_convs):
    for _ in range(n_convs):
        x = layers.Conv2D(filtros, 3, padding="same", activation="relu")(x)
    return layers.MaxPooling2D(2)(x)

entrada = keras.Input(shape=(224, 224, 3))
x = bloque_vgg(entrada, 64, 2)
x = bloque_vgg(x, 128, 2)
x = bloque_vgg(x, 256, 3)
vgg_mini = keras.Model(entrada, x, name="VGG_mini")
print("salida tras 3 bloques VGG:", vgg_mini.output_shape)   # (None, 28, 28, 256)

## 3. Bloque residual (ResNet): `y = x + F(x)`

El skip connection deja fluir el gradiente y permite redes muy profundas. Si la
conv cambia canales o resolución, la skip se proyecta con una conv `1×1`.

In [ ]:
def bloque_residual(x, filtros, stride=1):
    shortcut = x
    y = layers.Conv2D(filtros, 3, strides=stride,
                      padding="same", use_bias=False)(x)
    y = layers.BatchNormalization()(y)
    y = layers.Activation("relu")(y)
    y = layers.Conv2D(filtros, 3, padding="same", use_bias=False)(y)
    y = layers.BatchNormalization()(y)
    if stride != 1 or shortcut.shape[-1] != filtros:
        shortcut = layers.Conv2D(filtros, 1, strides=stride,
                                 use_bias=False)(shortcut)
        shortcut = layers.BatchNormalization()(shortcut)
    y = layers.Add()([shortcut, y])          # skip connection
    return layers.Activation("relu")(y)

entrada = keras.Input(shape=(56, 56, 64))
x = bloque_residual(entrada, 64)
x = bloque_residual(x, 128, stride=2)
resnet_mini = keras.Model(entrada, x, name="ResNet_mini")
print("salida:", resnet_mini.output_shape)   # (None, 28, 28, 128)

## 4. Depthwise-separable (Xception): ~10× menos parámetros

`SeparableConv2D` = conv por canal + conv `1×1` que mezcla canales. Mismo
receptive field con muchos menos parámetros que una `Conv2D` densa.

In [ ]:
std = keras.Sequential([keras.Input(shape=(32, 32, 64)),
                        layers.Conv2D(128, 3, padding="same")])
sep = keras.Sequential([keras.Input(shape=(32, 32, 64)),
                        layers.SeparableConv2D(128, 3, padding="same")])
print("Conv2D(128, 3)         :", std.count_params())    # 73856
print("SeparableConv2D(128, 3):", sep.count_params())    # 8896 (~8x menos)

## 5. Squeeze-and-Excite (SENet): attention sobre canales

Re-pesa los canales de forma aprendida:
`GlobalAvgPool → Dense(C/r) → Dense(C, sigmoid) → multiply`.

In [ ]:
def bloque_se(x, r=16):
    c = x.shape[-1]
    s = layers.GlobalAveragePooling2D()(x)             # squeeze
    s = layers.Dense(c // r, activation="relu")(s)      # reducción
    s = layers.Dense(c, activation="sigmoid")(s)        # pesos por canal [0,1]
    s = layers.Reshape((1, 1, c))(s)
    return layers.Multiply()([x, s])                    # excite: re-escala

entrada = keras.Input(shape=(28, 28, 256))
salida = bloque_se(entrada)
se = keras.Model(entrada, salida, name="SE_block")
print("SE block salida:", se.output_shape)   # (None, 28, 28, 256)

## 6. `keras.applications` y tabla comparativa

El catálogo trae las arquitecturas con pesos ImageNet. Aquí las construimos con
`weights=None` (solo la topología, sin descargar) y comparamos tamaños.

In [ ]:
resnet = keras.applications.ResNet50(weights=None, include_top=True)
effnet = keras.applications.EfficientNetB0(weights=None, include_top=True)
print(f"ResNet50       : {resnet.count_params():,} params")
print(f"EfficientNetB0 : {effnet.count_params():,} params")

comparativa = [
    ("LeNet-5",      1998, "primera CNN, dígitos"),
    ("AlexNet",      2012, "ReLU + Dropout + GPU"),
    ("VGG-16",       2014, "solo conv 3x3, 138M params"),
    ("GoogLeNet",    2014, "modulos Inception paralelos"),
    ("ResNet",       2015, "skip connections, hasta 152 capas"),
    ("Xception",     2017, "depthwise-separable"),
    ("SENet",        2017, "squeeze-and-excite (canales)"),
    ("EfficientNet", 2019, "compound scaling"),
    ("ConvNeXt",     2022, "ResNet modernizada con trucos ViT"),
]
for nombre, anio, nota in comparativa:
    print(f"{anio}  {nombre:<13} {nota}")

## Ejercicios

1. **LeNet-5**: construí la red y contá sus parámetros con `count_params()`.
2. **Mini-ResNet**: apilá 4 `bloque_residual` y verificá que `Add()` no falla
   cuando la skip se proyecta con conv `1×1`.
3. **Depthwise-separable**: compará los parámetros de `Conv2D(128, 3)` vs
   `SeparableConv2D(128, 3)` sobre la misma entrada.
4. **Squeeze-Excite**: aplicá `bloque_se` a un tensor `(28, 28, 256)` y
   verificá que la salida conserva el shape de entrada.

## Conclusiones

- Las CNN evolucionaron hacia más profundidad, paths múltiples y eficiencia.
- La **skip connection** de ResNet (`y = x + F(x)`) hizo entrenable lo muy profundo.
- **Depthwise-separable** (Xception) recorta ~10× los parámetros con igual receptive field.
- **Squeeze-and-excite** (SENet) reordena la importancia de los canales aprendida.
- Hoy `EfficientNet` y `ConvNeXt` son defaults fuertes; `keras.applications` los trae listos.